# M9 in Jupyter

Every cell is a whole module. A cell that begins `MODULE Name` is a program: it is checked, compiled and run, and what it prints appears below it. A cell that begins `DEFINITION MODULE Name` is a library: it is checked, and every later cell may `IMPORT` it.

Code is shared between cells; values are not. Data travels between cells through files, and a figure is a file: any SVG or PNG a cell writes is shown under it.

This notebook is also the kernel's gate (`runtime/test/jupyter.sh`), which runs it and checks every output.

In [ ]:
DEFINITION MODULE Kelvin ;

PROCEDURE FromCelsius (c: F64) : F64 ;

END Kelvin.

IMPLEMENTATION MODULE Kelvin ;

PROCEDURE FromCelsius (c: F64) : F64 =
BEGIN
  RETURN c + 273.15
END FromCelsius ;

END Kelvin.

In [ ]:
MODULE Temps ;

IMPORT Io ;
IMPORT Fmt ;
IMPORT Kelvin ;

BEGIN
  Io.WriteLine ('15 C is ' + Fmt.Fixed (Kelvin.FromCelsius (15.0), 2) + ' K')
EXCEPT
| ValueRange :
    Io.Halt (1)
END Temps.

In [ ]:
MODULE Damped ;

IMPORT Io ;
IMPORT Math ;
IMPORT Plot ;

VAR
  xs, ys, es : ARRAY 100 OF F64 ;
  i : I64 ;
  t : F64 ;

BEGIN
  FOR i := 0 TO 99 DO
    t := F64 (i) * 0.1 ;
    xs [i] := t ;
    es [i] := Math.Exp (0.0 - 0.3 * t) ;
    ys [i] := es [i] * Math.Cos (3.0 * t)
  END ;
  Plot.ClearFigure () ;
  Plot.AddLine (xs, ys, 0, 'signal') ;
  Plot.AddLine (xs, es, 1, 'envelope') ;
  Io.WriteFile ('damped.svg',
                Plot.Render ('damped oscillation', 'time (s)', 'amplitude'))
EXCEPT
| ValueRange :
    Io.ErrLine ('rendering failed') ; Io.Halt (1)
| Overflow :
    Io.ErrLine ('overflow in the series') ; Io.Halt (1)
| Io.IOError :
    Io.ErrLine ('cannot write damped.svg') ; Io.Halt (1)
END Damped.

Values cross from one cell to the next by name, through `NbCells`: one cell stores, a later one reads, and the kernel says what was stored. The files are Parquet in `NOTEBOOK.m9data` beside the notebook, so pandas reads them too.

In [ ]:
MODULE Store ;

IMPORT Io ;
IMPORT Math ;
IMPORT Faults ;
IMPORT NbCells ;

VAR
  xs : ARRAY 5 OF F64 ;
  i : I64 ;

BEGIN
  FOR i := 0 TO 4 DO xs [i] := Math.Sqrt (F64 (i)) END ;
  NbCells.PutF64s ('roots', xs)
EXCEPT
| ValueRange :
    Io.ErrLine ('a root failed') ; Io.Halt (1)
| Faults.BadArg (what) :
    Io.ErrLine ('not a name') ; Io.Halt (1)
| Faults.SizeError (got, want) :
    Io.ErrLine ('a size') ; Io.Halt (1)
| Io.IOError (path) :
    Io.ErrLine ('cannot write') ; Io.Halt (1)
END Store.

In [ ]:
MODULE Use ;

IMPORT Io ;
IMPORT Fmt ;
IMPORT Faults ;
IMPORT NbCells ;

VAR
  pool : POOL ;
  xs : SLICE OF F64 ;
  s : F64 ;
  i : I64 ;

BEGIN
  xs := NbCells.GetF64s (pool, 'roots') ;
  s := 0.0 ;
  FOR i := 0 TO LEN (xs) - 1 DO s := s + xs [i] END ;
  Io.WriteLine (Fmt.I64Str (LEN (xs)) + ' roots, sum ' + Fmt.Fixed (s, 6))
EXCEPT
| NbCells.Missing (name) :
    Io.ErrLine ('nothing stored as roots: run the cell above first') ; Io.Halt (1)
| NbCells.WrongType (name, stored, wanted) :
    Io.ErrLine ('roots holds something else') ; Io.Halt (1)
| ValueRange :
    Io.ErrLine ('formatting failed') ; Io.Halt (1)
| Faults.BadArg (what) :
    Io.ErrLine ('not a name') ; Io.Halt (1)
| Io.IOError (path) :
    Io.ErrLine ('cannot read') ; Io.Halt (1)
END Use.

A cell the checker refuses is not run. The diagnostic carries the cell's own line and column.

In [ ]:
MODULE Narrow ;

VAR x : I32 ;

BEGIN
  x := 40000000000
END Narrow.